# Setup

In [ ]:
import os
from openai import OpenAI  # ! - openai v1 SDK import; the old `import openai` + `openai.Completion` API was removed in v1

client = OpenAI()  # ! - v1 client, auto-reads OPENAI_API_KEY from env (replaces `openai.api_key = os.environ[...]`)

# ! - constrain the chat model to put Thought and Action on SEPARATE lines (what webthink's split() expects); the old 'single line' wording made gpt-4o-mini cram them onto one line, breaking the parse (the `ohh...` fallback)
SYSTEM = (  # ! -
    "You are a text-completion engine continuing a ReAct transcript. "  # ! -
    "Continue in the exact same format: output the next 'Thought N:' line, then a newline, "  # ! -
    "then the next 'Action N:' line, and then stop. "  # ! -
    "Keep the Thought and Action on separate lines; never put the Action on the same line as the Thought. "  # ! -
    "If you are not yet certain, still give your best-guess answer with Finish[...] before you run low on steps; never end without a Finish[...]. "  # ! - counteract RLHF hedging: gpt-4o-mini times out empty ~29% vs davinci ~2%
    "Never write an Observation line, never explain, never add prose or markdown."  # ! -
)  # ! -

def llm(prompt, stop=["\n"]):
    response = client.chat.completions.create(  # ! - v1 chat endpoint (was `openai.Completion.create`)
      model="gpt-4o-mini",  # ! - text-davinci-002 was retired by OpenAI on 2024-01-04
      messages=[  # ! - chat API takes a messages list, not a single `prompt=` string
        {"role": "system", "content": SYSTEM},  # ! -
        {"role": "user", "content": prompt},  # ! -
      ],  # ! -
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop + ["Observation", "Observation:"]  # ! - guard: stop before the model can fabricate an Observation line (those must come from the env)
    )
    return response.choices[0].message.content  # ! - v1 response object + chat message shape (was response["choices"][0]["text"])

In [ ]:
import requests  # ! - step() catches requests.exceptions.Timeout, but the notebook never imported requests -> any error became a NameError
import wikienv, wrappers
env = wikienv.WikiEnv()
env = wrappers.FeverWrapper(env, split="dev")
env = wrappers.LoggingWrapper(env)

def step(env, action):
    attempts = 0
    while attempts < 10:
        try:
            return env.step(action)
        except requests.exceptions.Timeout:
            attempts += 1

# ReAct

In [ ]:
import json
import sys

folder = './prompts/'
prompt_file = 'fever.json'
with open(folder + prompt_file, 'r') as f:
    prompt_dict = json.load(f)

webthink_prompt = prompt_dict['webthink_simple3']

def webthink(idx=None, prompt=webthink_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        n_calls += 1
        thought_action = llm(prompt + f"Thought {i}:", stop=[f"\nObservation {i}:"])
        try:
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        except:
            print('ohh...', thought_action)
            n_badcalls += 1
            n_calls += 1
            thought = thought_action.strip().split('\n')[0]
            action = llm(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=[f"\n"]).strip()
        obs, r, done, info = step(env, action[0].lower() + action[1:])
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        prompt += step_str
        if to_print:
            print(step_str)
        if done:
            break
    if not done:
        obs, r, done, info = step(env, "finish[]")
    if to_print:
        print(info, '\n')
    info.update({'n_calls': n_calls, 'n_badcalls': n_badcalls, 'traj': prompt})
    return r, info

In [ ]:
import random
import time
idxs = list(range(7405))
random.Random(233).shuffle(idxs)

rs = []
infos = []
old_time = time.time()
for i in idxs[:500]:
    try:  # ! - salvage: don't let one bad example kill the whole run; record it as failed and continue
        r, info = webthink(i, to_print=True)  # ! - re-indented (original line)
        rs.append(info['em'])  # ! - re-indented (original line)
        infos.append(info)  # ! - re-indented (original line)
    except Exception as e:  # ! -
        print(f"!! skipped idx {i}: {type(e).__name__}: {e}")  # ! -
        rs.append(0)  # ! -
        infos.append({'em': 0, 'question_idx': i, 'error': f"{type(e).__name__}: {e}"})  # ! -
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()